# 13 经典策略：双均线、布林带、横截面动量

## 13.1 本章目标

本章用三类经典策略把前面的工具再走一遍。重点不是寻找收益最高的参数，而是看清楚策略规则如何写成信号、权重、回测和订单建议。

三类策略分别是：

1. **双均线趋势**：用短均线和长均线判断趋势。
2. **布林带均值回归**：用价格偏离均值的程度判断反弹机会。
3. **横截面动量 TopN**：在多个 ETF 中选择近期表现更强的资产。

每个策略都保持规则可见，回测、报告和交易信号则复用 `lib`。

## 13.2 输入

- 来自 `data/sample/prices.parquet` 的真实 ETF 缓存行情。
- 用 `close.pct_change()` 计算收益，并用 `shift(1)` 保证信号滞后生效。
- 复用第 08 章的 TopN 权重思路，以及第 09-10 章的回测和报告口径。

## 13.3 输出

每个策略会写入自己的结果目录：

- `outputs/results/classic_strategies/ma_cross/`
- `outputs/results/classic_strategies/bollinger_reversion/`
- `outputs/results/classic_strategies/cross_section_momentum/`

这些输出包括绩效指标、净值图、回撤图、目标权重和订单建议。


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    candidates += [candidate / "pyquant-roadmap" for candidate in candidates]
    for candidate in candidates:
        if (candidate / "lib").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise RuntimeError("Cannot find pyquant-roadmap project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.backtest import returns_from_weights
from lib.data import load_sample_assets, load_sample_prices
from lib.evaluation.metrics import perf_stats
from lib.paths import RESULTS_DIR
from lib.portfolio import top_n_equal_weight, weights_to_matrix
from lib.reporting import save_strategy_results
from lib.trading import latest_target_weights, target_weights_to_orders

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

RESULT_BASE = RESULTS_DIR / "classic_strategies"
RESULT_BASE.mkdir(parents=True, exist_ok=True)

ASSET = "510300"
TOP_N = 2
COST_BPS = 8.0
CAPITAL = 1_000_000
LOT_SIZE = 100

prices = load_sample_prices().copy()
prices["date"] = pd.to_datetime(prices["date"])
prices["code"] = prices["code"].astype(str)

assets = load_sample_assets().copy()
assets["code"] = assets["code"].astype(str)

close = prices.pivot(index="date", columns="code", values="close").sort_index().astype(float)
asset_names = assets.set_index("code")["name"].to_dict()

summary = pd.DataFrame(
    {
        "start": close.apply(lambda s: s.first_valid_index()),
        "end": close.apply(lambda s: s.last_valid_index()),
        "rows": close.notna().sum(),
        "missing": close.isna().sum(),
        "latest_close": close.iloc[-1],
    }
).reset_index().rename(columns={"code": "code"})
summary.insert(1, "name", summary["code"].map(asset_names))

print("Project root: .")
print(f"Cached ETF close matrix: {close.shape[0]} trading days x {close.shape[1]} assets")
display(summary)


Project root: .
Cached ETF close matrix: 725 trading days x 4 assets


,code,name,start,end,rows,missing,latest_close
0,159915,创业板ETF,2021-01-04,2023-12-29,725,0,1.842
1,510300,沪深300ETF,2021-01-04,2023-12-29,725,0,3.219
2,510500,中证500ETF,2021-01-04,2023-12-29,725,0,5.279
3,512100,中证1000ETF,2021-01-04,2023-12-29,725,0,2.296


## 13.4 共用规则：先产生目标权重，再回测

三个策略都遵守同一条原则：**在日期 `t` 形成的目标权重，只能从 `t+1` 的收益开始体现**。`lib.backtest.returns_from_weights` 内部使用 `weights.shift(1)`，因此能保持统一口径。

本章所有信号都采用 point-in-time safe 写法：先把收盘价滞后一日，再计算均线、布林带或动量分数。

先准备几个共用函数。


In [2]:
def single_asset_target_weights(signal: pd.Series, columns: pd.Index, asset: str) -> pd.DataFrame:
    """Convert a 0/1 single-asset signal into a full target-weight matrix."""
    weights = pd.DataFrame(0.0, index=signal.index, columns=columns)
    weights[asset] = signal.reindex(signal.index).fillna(0.0).clip(lower=0.0, upper=1.0)
    return weights


def month_end_trading_dates(dates) -> pd.DatetimeIndex:
    """Return the last available trading date in each calendar month."""
    unique_dates = pd.Series(pd.to_datetime(pd.Index(dates).unique())).sort_values()
    month_end = unique_dates.groupby(unique_dates.dt.to_period("M")).max()
    return pd.DatetimeIndex(month_end.to_list())


def describe_weight_matrix(weights: pd.DataFrame) -> pd.DataFrame:
    active_counts = weights.gt(0).sum(axis=1)
    return pd.DataFrame(
        {
            "first_weight_date": [weights.index[weights.sum(axis=1).gt(0)].min() if weights.sum(axis=1).gt(0).any() else pd.NaT],
            "latest_weight_date": [weights.index.max() if not weights.empty else pd.NaT],
            "latest_invested_weight": [weights.iloc[-1].sum() if not weights.empty else np.nan],
            "max_assets_held": [active_counts.max() if not weights.empty else 0],
            "rebalance_like_days": [weights.diff().abs().sum(axis=1).gt(0).sum() if len(weights) else 0],
        }
    )


## 13.5 策略一：双均线趋势

### 13.5.1 规则

选择单只 ETF，这里用 `510300` 做示例：

- 用昨日以前的数据计算短期均线 `MA_short` 和长期均线 `MA_long`。
- 当 `MA_short > MA_long` 时，目标权重为 100%。
- 否则目标权重为 0%。
- 信号生成后交给统一回测函数。

这是趋势跟随策略的最小版本。


In [3]:
def dual_ma_trend_signal(close_s: pd.Series, short_window: int = 20, long_window: int = 60) -> pd.DataFrame:
    known_close = close_s.shift(1)
    short_ma = known_close.rolling(short_window).mean()
    long_ma = known_close.rolling(long_window).mean()
    signal = (short_ma > long_ma).astype(float)
    signal[long_ma.isna()] = 0.0
    return pd.DataFrame(
        {
            "close": close_s,
            f"ma_{short_window}": short_ma,
            f"ma_{long_window}": long_ma,
            "signal": signal,
        }
    )


ma_signal_table = dual_ma_trend_signal(close[ASSET], short_window=20, long_window=60)
ma_weights = single_asset_target_weights(ma_signal_table["signal"], close.columns, ASSET)

print("Dual moving-average signal sample:")
display(ma_signal_table.dropna().tail(8).round(4))
print("Target-weight matrix check:")
display(describe_weight_matrix(ma_weights))
display(ma_weights.tail().round(4))


Dual moving-average signal sample:


,close,ma_20,ma_60,signal
date,,,,
2023-12-20,3.089,3.2241,3.3423,0.0
2023-12-21,3.120,3.2120,3.3352,0.0
2023-12-22,3.126,3.2005,3.3289,0.0
2023-12-25,3.135,3.1906,3.3233,0.0
2023-12-26,3.115,3.1823,3.3166,0.0
2023-12-27,3.125,3.1727,3.3100,0.0
2023-12-28,3.209,3.1651,3.3040,0.0
2023-12-29,3.219,3.1614,3.2992,0.0


Target-weight matrix check:


,first_weight_date,latest_weight_date,latest_invested_weight,max_assets_held,rebalance_like_days
0,2021-05-26,2023-12-29,0.0,1,16


code,159915,510300,510500,512100
date,,,,
2023-12-25,0.0,0.0,0.0,0.0
2023-12-26,0.0,0.0,0.0,0.0
2023-12-27,0.0,0.0,0.0,0.0
2023-12-28,0.0,0.0,0.0,0.0
2023-12-29,0.0,0.0,0.0,0.0


## 13.6 策略二：布林带均值回归

### 13.6.1 规则

仍然使用 `510300` 做单资产示例：

- 用昨日以前的数据计算中轨 `mid = rolling_mean(window)`。
- 下轨为 `lower = mid - k * rolling_std(window)`。
- 当已知价格低于下轨时，认为短期偏离较大，目标权重为 100%。
- 当已知价格回到中轨以上时，目标权重降为 0%。
- 其他日期延续上一期状态。

这是均值回归策略的最小状态机。


In [4]:
def bollinger_reversion_signal(close_s: pd.Series, window: int = 20, k: float = 2.0) -> pd.DataFrame:
    known_close = close_s.shift(1)
    mid = known_close.rolling(window).mean()
    std = known_close.rolling(window).std(ddof=0)
    upper = mid + k * std
    lower = mid - k * std

    state_change = pd.Series(np.nan, index=close_s.index, dtype=float)
    state_change[known_close < lower] = 1.0
    state_change[known_close >= mid] = 0.0
    signal = state_change.ffill().fillna(0.0)
    signal[mid.isna()] = 0.0

    return pd.DataFrame(
        {
            "close": close_s,
            "known_close": known_close,
            "mid": mid,
            "upper": upper,
            "lower": lower,
            "state_change": state_change,
            "signal": signal,
        }
    )


bb_signal_table = bollinger_reversion_signal(close[ASSET], window=20, k=2.0)
bb_weights = single_asset_target_weights(bb_signal_table["signal"], close.columns, ASSET)

changes = bb_signal_table[bb_signal_table["state_change"].notna()].tail(8)
print("Recent Bollinger state changes:")
display(changes.round(4))
print("Target-weight matrix check:")
display(describe_weight_matrix(bb_weights))
display(bb_weights.tail().round(4))


Recent Bollinger state changes:


,close,known_close,mid,upper,lower,state_change,signal
date,,,,,,,
2023-11-21,3.370,3.365,3.3626,3.4379,3.2873,0.0,0.0
2023-11-22,3.331,3.370,3.3676,3.4295,3.3058,0.0,0.0
2023-11-28,3.305,3.301,3.3670,3.4243,3.3098,1.0,1.0
2023-11-30,3.282,3.278,3.3599,3.4335,3.2863,1.0,1.0
2023-12-05,3.178,3.244,3.3426,3.4375,3.2478,1.0,1.0
2023-12-06,3.183,3.178,3.3310,3.4449,3.2172,1.0,1.0
2023-12-07,3.174,3.183,3.3202,3.4466,3.1939,1.0,1.0
2023-12-29,3.219,3.209,3.1614,3.2549,3.0680,0.0,0.0


Target-weight matrix check:


,first_weight_date,latest_weight_date,latest_invested_weight,max_assets_held,rebalance_like_days
0,2021-03-09,2023-12-29,0.0,1,32


code,159915,510300,510500,512100
date,,,,
2023-12-25,0.0,1.0,0.0,0.0
2023-12-26,0.0,1.0,0.0,0.0
2023-12-27,0.0,1.0,0.0,0.0
2023-12-28,0.0,1.0,0.0,0.0
2023-12-29,0.0,0.0,0.0,0.0


## 13.7 策略三：横截面动量 TopN

### 13.7.1 规则

横截面动量走的是“评分 -> TopN 选择 -> 组合权重”路线：

- 对每只 ETF 计算 `lookback` 日动量，且只使用 `t-1` 之前已知的价格。
- 每个调仓日在同一资产池内比较分数。
- 选择分数最高的 TopN。
- TopN 内等权，未入选资产权重为 0%。
- 权重持有到下一个调仓日。

这和第 06-08 章的多因子主线一致，只是这里使用单一动量分数。


In [5]:
def cross_section_momentum_score(price: pd.DataFrame, lookback: int = 60) -> pd.DataFrame:
    known_close = price.shift(1)
    return known_close / known_close.shift(lookback) - 1.0


def topn_weights_manual(score_matrix: pd.DataFrame, top_n: int = 2) -> tuple[pd.DataFrame, pd.DataFrame]:
    rebalance_dates = month_end_trading_dates(score_matrix.index)
    rows = []
    for dt in rebalance_dates:
        scores = score_matrix.loc[dt].dropna().rename("score").reset_index().rename(columns={"index": "code"})
        if scores.empty:
            continue
        top = scores.sort_values(["score", "code"], ascending=[False, True], kind="mergesort").head(top_n)
        weight = 1.0 / len(top)
        top = top.assign(date=dt, weight=weight)
        rows.append(top[["date", "code", "score", "weight"]])

    sparse = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["date", "code", "score", "weight"])
    matrix = weights_to_matrix(sparse[["date", "code", "weight"]], index=score_matrix.index, columns=score_matrix.columns, carry_forward=True)
    return sparse.sort_values(["date", "code"]).reset_index(drop=True), matrix


momentum_score = cross_section_momentum_score(close, lookback=60)
mom_sparse_manual, mom_weights = topn_weights_manual(momentum_score, top_n=TOP_N)

print("Latest monthly TopN selections:")
display(mom_sparse_manual.tail(10).merge(assets[["code", "name"]], on="code", how="left").round({"score": 4, "weight": 4}))
print("Target-weight matrix check:")
display(describe_weight_matrix(mom_weights))
display(mom_weights.tail().round(4))


Latest monthly TopN selections:


,date,code,score,weight,name
0,2023-08-31,159915,-0.0359,0.5,创业板ETF
1,2023-08-31,510300,0.0036,0.5,沪深300ETF
2,2023-09-28,510300,-0.0356,0.5,沪深300ETF
3,2023-09-28,510500,-0.0551,0.5,中证500ETF
4,2023-10-31,510500,-0.0874,0.5,中证500ETF
5,2023-10-31,512100,-0.0749,0.5,中证1000ETF
6,2023-11-30,510500,-0.0387,0.5,中证500ETF
7,2023-11-30,512100,0.0029,0.5,中证1000ETF
8,2023-12-29,510500,-0.0557,0.5,中证500ETF
9,2023-12-29,512100,-0.0387,0.5,中证1000ETF


Target-weight matrix check:


,first_weight_date,latest_weight_date,latest_invested_weight,max_assets_held,rebalance_like_days
0,2021-04-30,2023-12-29,1.0,2,20


code,159915,510300,510500,512100
date,,,,
2023-12-25,0.0,0.0,0.5,0.5
2023-12-26,0.0,0.0,0.5,0.5
2023-12-27,0.0,0.0,0.5,0.5
2023-12-28,0.0,0.0,0.5,0.5
2023-12-29,0.0,0.0,0.5,0.5


### 13.7.2 复用第 08 章的组合工具

手写 TopN 有助于理解规则，但实战里应该复用第 08 章沉淀的 `lib.portfolio`：

- `top_n_equal_weight(scores, n, score_col)`：把 `date, code, score` 长表转成 TopN 稀疏权重。
- `weights_to_matrix(weights, index, columns, carry_forward=True)`：把调仓权重展开成每日目标权重矩阵。

下面检查手写实现和 `lib` 实现是否一致。


In [6]:
rebalance_dates = month_end_trading_dates(momentum_score.index)
momentum_score_long = (
    momentum_score.loc[rebalance_dates]
    .stack(future_stack=True)
    .rename("score")
    .reset_index()
)
momentum_score_long.columns = ["date", "code", "score"]

mom_sparse_lib = top_n_equal_weight(momentum_score_long, n=TOP_N, score_col="score")
mom_weights_lib = weights_to_matrix(mom_sparse_lib, index=close.index, columns=close.columns, carry_forward=True)

check = (mom_weights_lib - mom_weights).abs().sum().sum()
print(f"Manual vs lib TopN weight difference: {check:.10f}")
display(mom_sparse_lib.tail(10).merge(assets[["code", "name"]], on="code", how="left"))

# 后续统一使用 lib 生成的权重矩阵，避免手写实现和实战实现分叉。
mom_weights = mom_weights_lib


Manual vs lib TopN weight difference: 0.0000000000


,date,code,weight,name
0,2023-08-31,159915,0.5,创业板ETF
1,2023-08-31,510300,0.5,沪深300ETF
2,2023-09-28,510300,0.5,沪深300ETF
3,2023-09-28,510500,0.5,中证500ETF
4,2023-10-31,510500,0.5,中证500ETF
5,2023-10-31,512100,0.5,中证1000ETF
6,2023-11-30,510500,0.5,中证500ETF
7,2023-11-30,512100,0.5,中证1000ETF
8,2023-12-29,510500,0.5,中证500ETF
9,2023-12-29,512100,0.5,中证1000ETF


## 13.8 统一回测、报告和交易信号输出

三个策略共用同一套实战内核：

1. `lib.backtest.returns_from_weights` 计算收益、成本、换手和 NAV。
2. `lib.reporting.save_strategy_results` 保存指标、图表、目标权重和订单建议。
3. `lib.trading.latest_target_weights` 与 `target_weights_to_orders` 生成最新权重和订单建议。

这说明经典策略规则可以写在 notebook 中，重复的工程能力则沉淀到 `lib`。


In [7]:
def equal_weight_benchmark(price: pd.DataFrame) -> pd.Series:
    weights = pd.DataFrame(1.0 / price.shape[1], index=price.index, columns=price.columns)
    return (weights.shift(1).fillna(0.0) * price.pct_change().fillna(0.0)).sum(axis=1)


strategies = {
    "ma_cross": {
        "label": "Dual moving-average trend",
        "weights": ma_weights,
        "benchmark": close[ASSET].pct_change().fillna(0.0).rename(f"{ASSET}_buy_hold"),
    },
    "bollinger_reversion": {
        "label": "Bollinger-band mean reversion",
        "weights": bb_weights,
        "benchmark": close[ASSET].pct_change().fillna(0.0).rename(f"{ASSET}_buy_hold"),
    },
    "cross_section_momentum": {
        "label": f"Cross-sectional momentum Top{TOP_N}",
        "weights": mom_weights,
        "benchmark": equal_weight_benchmark(close).rename("equal_weight_all_etf"),
    },
}

summary_rows = []
result_paths = {}
latest_signal_tables = []

for name, config in strategies.items():
    weights = config["weights"].reindex(index=close.index, columns=close.columns).fillna(0.0)
    report = returns_from_weights(close, weights, cost_bps=COST_BPS)
    latest = latest_target_weights(weights).merge(assets[["code", "name"]], on="code", how="left")
    orders = target_weights_to_orders(weights, close, capital=CAPITAL, lot_size=LOT_SIZE)
    if not orders.empty:
        orders = orders.merge(assets[["code", "name"]], on="code", how="left")

    output_dir = RESULT_BASE / name
    paths = save_strategy_results(
        returns=report["strategy_return"],
        nav=report["nav"],
        target_weights=latest,
        orders=orders,
        output_dir=output_dir,
        benchmark_returns=config["benchmark"],
    )
    paths["target_weight_matrix"] = output_dir / "target_weight_matrix.csv"
    paths["strategy_returns"] = output_dir / "strategy_returns.csv"
    weights.to_csv(paths["target_weight_matrix"], index_label="date", encoding="utf-8-sig")
    report.to_csv(paths["strategy_returns"], index_label="date", encoding="utf-8-sig")

    stats = perf_stats(report["strategy_return"], config["benchmark"])
    summary_rows.append(
        {
            "strategy": name,
            "label": config["label"],
            "latest_invested_weight": weights.iloc[-1].sum(),
            "avg_turnover": report["turnover"].mean(),
            **stats,
        }
    )
    latest_signal_tables.append(latest.assign(strategy=name, label=config["label"]))
    result_paths[name] = {key: str(value.relative_to(PROJECT_ROOT)) for key, value in paths.items()}

classic_summary = pd.DataFrame(summary_rows).set_index("strategy")
classic_summary_path = RESULT_BASE / "classic_summary.csv"
classic_summary.to_csv(classic_summary_path, encoding="utf-8-sig")

numeric_cols = classic_summary.select_dtypes(include="number").columns
classic_summary_display = classic_summary.copy()
classic_summary_display[numeric_cols] = classic_summary_display[numeric_cols].round(4)

display(classic_summary_display)


,label,latest_invested_weight,avg_turnover,ann_return,ann_vol,sharpe,max_drawdown,total_return,win_rate,benchmark_total_return,benchmark_ann_return,excess_total_return,excess_ann_return,active_return,tracking_error,information_ratio
strategy,,,,,,,,,,,,,,,,
ma_cross,Dual moving-average trend,0.0,0.0221,-0.0615,0.0867,-0.6886,-0.1907,-0.1670,0.1476,-0.3353,-0.1324,0.1683,0.0708,0.1522,0.1709,0.3737
bollinger_reversion,Bollinger-band mean reversion,0.0,0.0441,-0.0035,0.1209,0.0317,-0.1845,-0.0100,0.1641,-0.3353,-0.1324,0.3254,0.1289,0.3976,0.1485,0.8578
cross_section_momentum,Cross-sectional momentum Top2,1.0,0.0276,-0.1350,0.1855,-0.6883,-0.4557,-0.3412,0.4386,-0.2355,-0.0891,-0.1056,-0.0459,-0.1500,0.0901,-0.5817


In [8]:
latest_signals = pd.concat(latest_signal_tables, ignore_index=True)
latest_signals_display = latest_signals.copy()
latest_signals_display["date"] = pd.to_datetime(latest_signals_display["date"]).dt.date
latest_signals_display = latest_signals_display.sort_values(["strategy", "target_weight"], ascending=[True, False])

display(latest_signals_display.round({"target_weight": 4}))


,date,code,target_weight,name,strategy,label
0,2023-12-29,510500,0.5,中证500ETF,cross_section_momentum,Cross-sectional momentum Top2
1,2023-12-29,512100,0.5,中证1000ETF,cross_section_momentum,Cross-sectional momentum Top2


In [9]:
path_rows = []
for strategy, artifacts in result_paths.items():
    for artifact, rel_path in artifacts.items():
        path_rows.append({"strategy": strategy, "artifact": artifact, "path": rel_path})
path_rows.append(
    {
        "strategy": "all",
        "artifact": "classic_summary",
        "path": str(classic_summary_path.relative_to(PROJECT_ROOT)),
    }
)
path_table = pd.DataFrame(path_rows).sort_values(["strategy", "artifact"]).reset_index(drop=True)
display(path_table)


,strategy,artifact,path
0,all,classic_summary,outputs\results\classic_strategies\classic_sum...
1,bollinger_reversion,benchmark_comparison,outputs\results\classic_strategies\bollinger_r...
2,bollinger_reversion,drawdown_curve,outputs\results\classic_strategies\bollinger_r...
3,bollinger_reversion,metrics_csv,outputs\results\classic_strategies\bollinger_r...
4,bollinger_reversion,metrics_md,outputs\results\classic_strategies\bollinger_r...
5,bollinger_reversion,nav_curve,outputs\results\classic_strategies\bollinger_r...
6,bollinger_reversion,strategy_returns,outputs\results\classic_strategies\bollinger_r...
7,bollinger_reversion,target_weight_matrix,outputs\results\classic_strategies\bollinger_r...
8,bollinger_reversion,target_weights,outputs\results\classic_strategies\bollinger_r...
9,bollinger_reversion,trade_orders,outputs\results\classic_strategies\bollinger_r...


## 13.9 质量检查和限制

至少检查以下事项：

- 目标权重不超过 100%。
- 权重没有负数。
- 单资产策略最多只持有一只 ETF。
- 横截面动量最多持有 TopN 只 ETF。
- 所需输出文件已经生成。

这些检查只保证流程和口径合理，不代表策略具有稳定收益。


In [10]:
checks = []
for name, config in strategies.items():
    weights = config["weights"].reindex(index=close.index, columns=close.columns).fillna(0.0)
    output_dir = RESULT_BASE / name
    required_files = [
        "performance_metrics.csv",
        "performance_metrics.md",
        "nav_curve.png",
        "drawdown_curve.png",
        "target_weights.csv",
        "trade_orders.csv",
        "target_weight_matrix.csv",
        "strategy_returns.csv",
    ]
    checks.append(
        {
            "strategy": name,
            "max_total_weight": weights.sum(axis=1).max(),
            "min_weight": weights.min().min(),
            "max_active_assets": weights.gt(0).sum(axis=1).max(),
            "required_files_exist": all((output_dir / file_name).exists() for file_name in required_files),
        }
    )

check_table = pd.DataFrame(checks)
check_table["passes_basic_checks"] = (
    check_table["max_total_weight"].le(1.0 + 1e-9)
    & check_table["min_weight"].ge(-1e-12)
    & check_table["required_files_exist"]
)

display(check_table.round({"max_total_weight": 4, "min_weight": 4}))

if not check_table["passes_basic_checks"].all():
    raise AssertionError("At least one strategy failed the basic notebook checks.")


,strategy,max_total_weight,min_weight,max_active_assets,required_files_exist,passes_basic_checks
0,ma_cross,1.0,0.0,1,True,True
1,bollinger_reversion,1.0,0.0,1,True,True
2,cross_section_momentum,1.0,0.0,2,True,True


## 13.10 小练习：把 TopN 从 Top2 改成 Top3

把 `TOP_N = 2` 改成 `TOP_N = 3` 后重新运行本章，观察：

1. `cross_section_momentum` 的持仓数量是否增加。
2. `target_weight_matrix.csv` 中每只入选 ETF 的权重是否接近 `1/3`。
3. 换手率和净值曲线是否发生变化。

这个练习会把 TopN 参数和第 07 章因子检验、第 11 章流水线参数联系起来。


## 13.11 本章小结

三类经典策略都可以落到同一条工程链路：

`规则 -> signal/score -> target weights -> returns/NAV -> report -> latest weights/orders`

需要注意：

- 本章只使用 4 只 ETF、2021-2023 年缓存样例数据。
- 回测采用简化收盘价成交和成本假设。
- 输出结果是学习材料，不是收益承诺，也不是投资建议。
- 策略规则写在 notebook 中，回测、报告和信号输出复用 `lib`。

第 14 章会收束学习路线，并讨论如何安全地使用 AI 辅助研究和检查。
